In [1]:
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,recall_score,precision_score
import re
import nltk
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

c:\Users\ASUS\Desktop\mini-project\Sentiment-Analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
df=pd.read_csv('https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv')


In [3]:
df['num_of_char']=df.sentiment.apply(lambda x: len(x))
df.head()

,tweet_id,sentiment,content,num_of_char
0,1956967341,empty,@tiffanylue i know i was listenin to bad habi...,5
1,1956967666,sadness,Layin n bed with a headache ughhhh...waitin o...,7
2,1956967696,sadness,Funeral ceremony...gloomy friday...,7
3,1956967789,enthusiasm,wants to hang out with friends SOON!,10
4,1956968416,neutral,@dannycastillo We want to trade with someone w...,7


In [4]:
df=df.drop('tweet_id',axis=1)

In [5]:
df.head(2)

,sentiment,content,num_of_char
0,empty,@tiffanylue i know i was listenin to bad habi...,5
1,sadness,Layin n bed with a headache ughhhh...waitin o...,7


In [6]:
#Data Preprocessing
def lemmatization(text):
    Text=[]
    lemmatizer=WordNetLemmatizer()
    for i in text.split():
        Text.append(lemmatizer.lemmatize(i))
    return ' '.join(Text)
        
def remove_stopwords(text):
    words=stopwords.words('english')
    text=[i for i in text.split() if i not in words]
    return ' '.join(text)

def removing_numbers(text):
    text=[i for i in text if not i.isdigit()]
    return ''.join(text)
    
def lower_case(text):
    return text.lower()
import string
def remove_punctuation(text):
    text=re.sub('[%s]' % re.escape(string.punctuation),' ',text)
    text=text.replace(':','')
    text=re.sub('/s+',' ',text).strip()
    return text

In [7]:
def normalize_text(df):
    df['content']=df['content'].apply(remove_punctuation)
    df['content']=df['content'].apply(removing_numbers)
    df['content']=df['content'].apply(lower_case)
    df['content']=df['content'].apply(remove_stopwords)
    df['content']=df['content'].apply(lemmatization)
    return df


In [ ]:
df['sentiment'].value_counts()
x=df['sentiment'].isin(['happiness','sadness'])
df=df[x]

In [27]:
map={}
enc=0
for i in df.sentiment.unique():
    map[i]=enc
    enc+=1
print(map)

{'sadness': 0, 'happiness': 1}


In [19]:
df['sentiment']=df['sentiment'].map({'happiness':0,'sadness':1})

In [9]:
df=normalize_text(df)

In [10]:
vectorizer=CountVectorizer(max_features=1000)
x=vectorizer.fit_transform(df['content'])

In [21]:

y=df['sentiment']
y.head()

1    1
2    1
6    1
8    1
9    1
Name: sentiment, dtype: int64

In [22]:
X_train,X_test,y_train,y_test=train_test_split(x,y,random_state=1,test_size=0.2)


In [23]:
import dagshub

dagshub.init(repo_owner='kanhacoderx',repo_name='MoodSense-AI',mlflow=True)
mlflow.set_tracking_uri('https://dagshub.com/kanhacoderx/MoodSense-AI.mlflow/')

Initialized MLflow to track repo "kanhacoderx/MoodSense-AI"

Repository kanhacoderx/MoodSense-AI initialized!

In [15]:
# mlflow.create_experiment('Logistic Regression baseline')
mlflow.set_experiment('Logistic Regression baseline')

<Experiment: artifact_location='mlflow-artifacts:/78afedc9fed140c486f2c7675d4f269c', creation_time=1778061883734, experiment_id='0', last_update_time=1778061883734, lifecycle_stage='active', name='Logistic Regression baseline', tags={}, trace_location=None, workspace='default'>

In [24]:
with mlflow.start_run():

    #Log PreProcess Param
    mlflow.log_param('vectorizer','Bag Of Words')
    mlflow.log_param('Max_Features',1000)
    mlflow.log_param('test_size',0.2)

    #Model Training
    model=LogisticRegression()
    model.fit(X_train,y_train)

    #Log Model Param
    mlflow.log_param('Model','Logistic Regression')


    #Model Evaluation
    y_pred=model.predict(X_test)

    acc=accuracy_score(y_pred,y_test)
    precision=precision_score(y_pred,y_test)
    recall=recall_score(y_test,y_pred)

    #Log Model 
    mlflow.sklearn.log_model(model,'Model')

    #Save Notebok
    import os
    notebook_path='exp1_baseline_model.ipynb'
    os.system('jupyter nbconvert --to notebook --execute --inplace {notebook_path}')
    mlflow.log_artifact(notebook_path)

    print('Accuracy',acc)
    metrics={
        'Precision':precision,
        'Accuracy':acc,
        'Recall':recall
    }
    print(metrics)
    #Metric Logging
    mlflow.log_metrics(metrics=metrics)


2026/05/06 17:36:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 17:36:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Accuracy 0.7816867469879518
{'Precision': 0.7872340425531915, 'Accuracy': 0.7816867469879518, 'Recall': 0.7872340425531915}
🏃 View run selective-donkey-96 at: https://dagshub.com/kanhacoderx/MoodSense-AI.mlflow/#/experiments/0/runs/620d00d5d92f471091448aa2a84e1b08
🧪 View experiment at: https://dagshub.com/kanhacoderx/MoodSense-AI.mlflow/#/experiments/0
